# Pre-label `dataset_enemy_icons` (`img####.png` only)
Loads the latest model checkpoint and prints top-1 prediction + probability.
This notebook does not rename or modify files.

In [1]:
from pathlib import Path
import re
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models

In [2]:
PROJECT_ROOT = Path("../")
CKPT_DIR = PROJECT_ROOT / "checkpoints"
DATASET_DIR = PROJECT_ROOT / "dataset_enemy_icons"

IMG_RE = re.compile(r"^img\d{4}\.png$", re.IGNORECASE)

ckpts = sorted(CKPT_DIR.glob("unit_resnet18_*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
if not ckpts:
    ckpts = sorted(CKPT_DIR.glob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert ckpts, f"No checkpoint found in {CKPT_DIR}"
ckpt_path = ckpts[0]
print("Using checkpoint:", ckpt_path)

img_paths = sorted([p for p in DATASET_DIR.glob("*.png") if IMG_RE.match(p.name)])
assert img_paths, f"No img####.png files found in {DATASET_DIR}"
print(f"Found {len(img_paths)} target files")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
payload = torch.load(ckpt_path, map_location=device)
class_names = payload["class_names"]
image_size = int(payload.get("image_size", 128))
mean = np.array(payload.get("normalize_mean", [0.5, 0.5, 0.5]), dtype=np.float32)
std = np.array(payload.get("normalize_std", [0.5, 0.5, 0.5]), dtype=np.float32)
std = np.where(std == 0, 1.0, std)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(payload["model_state_dict"], strict=True)
model = model.to(device)
model.eval()

def preprocess_icon_bgr(img):
    resized = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - mean) / std
    chw = np.transpose(rgb, (2, 0, 1))
    return torch.from_numpy(chw).unsqueeze(0).float().to(device)

for p in img_paths:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        print(f"{p.name} | READ_FAIL | prob=0.0000")
        continue

    inp = preprocess_icon_bgr(img)
    with torch.no_grad():
        logits = model(inp)
        probs = torch.softmax(logits, dim=1)[0]

    v, i = torch.max(probs, dim=0)
    print(f"{p.name} | {class_names[int(i)]} | prob={float(v):.4f}")


Using checkpoint: ..\checkpoints\unit_resnet18_20260316_013924.pt
Found 279 target files
img0001.png | The_Reaper_of_Life | prob=0.5888
img0002.png | Aegisriel | prob=0.6793
img0003.png | Crimson_Scorpion | prob=0.7078
img0004.png | Red_Goblin | prob=0.6732
img0005.png | Gator_Crocodile | prob=0.9285
img0006.png | Fire_Mage | prob=0.8633
img0007.png | Forbidden_Anubish | prob=0.8991
img0008.png | Tengu | prob=0.4563
img0009.png | Burning_Rock | prob=0.6444
img0010.png | Blaze_Steed | prob=0.4962
img0011.png | Rigamia | prob=0.9023
img0012.png | Bird_of_Domination | prob=0.9407
img0013.png | Charred_Lizard | prob=0.7376
img0014.png | Lamp_Spirit | prob=0.7844
img0015.png | Necro_Princess | prob=0.5497
img0016.png | Rainbow_Pelican_EXP | prob=0.9304
img0017.png | Mysterious_Jar | prob=0.8726
img0018.png | Creature_Rose | prob=0.9194
img0019.png | Bone_Battler | prob=0.8881
img0020.png | ChronoZelius | prob=0.8272
img0021.png | Primitive_Butterfly | prob=0.6664
img0022.png | Goblin_Mage |

In [3]:
ACCEPT_THRESHOLD = 0.40

def next_labeled_name(folder: Path, class_name: str):
    pat = re.compile(rf"^{re.escape(class_name)}_(\\d+)\\.png$", re.IGNORECASE)
    max_idx = 0
    for p in folder.iterdir():
        if not p.is_file():
            continue
        m = pat.match(p.name)
        if m:
            max_idx = max(max_idx, int(m.group(1)))

    idx = max_idx + 1
    while True:
        cand = f"{class_name}_{idx:04d}.png"
        if not (folder / cand).exists():
            return cand
        idx += 1

renamed = 0
kept = 0

img_paths = sorted([p for p in DATASET_DIR.glob("*.png") if IMG_RE.match(p.name)])
for p in img_paths:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        print(f"KEEP {p.name} | READ_FAIL")
        kept += 1
        continue

    inp = preprocess_icon_bgr(img)
    with torch.no_grad():
        logits = model(inp)
        probs = torch.softmax(logits, dim=1)[0]

    v, i = torch.max(probs, dim=0)
    prob = float(v)
    pred_name = class_names[int(i)]

    if prob >= ACCEPT_THRESHOLD:
        new_name = next_labeled_name(DATASET_DIR, pred_name)
        new_path = DATASET_DIR / new_name
        p.rename(new_path)
        print(f"RENAME {p.name} -> {new_name} | prob={prob:.4f}")
        renamed += 1
    else:
        print(f"KEEP {p.name} | pred={pred_name} | prob={prob:.4f}")
        kept += 1

print(f"done | renamed={renamed} kept={kept}")


RENAME img0001.png -> The_Reaper_of_Life_0013.png | prob=0.5888
RENAME img0002.png -> Aegisriel_0009.png | prob=0.6793
RENAME img0003.png -> Crimson_Scorpion_0012.png | prob=0.7078
RENAME img0004.png -> Red_Goblin_0009.png | prob=0.6732
RENAME img0005.png -> Gator_Crocodile_0013.png | prob=0.9285
RENAME img0006.png -> Fire_Mage_0006.png | prob=0.8633
RENAME img0007.png -> Forbidden_Anubish_0009.png | prob=0.8991
RENAME img0008.png -> Tengu_0016.png | prob=0.4563
RENAME img0009.png -> Burning_Rock_0011.png | prob=0.6444
RENAME img0010.png -> Blaze_Steed_0003.png | prob=0.4962
RENAME img0011.png -> Rigamia_0014.png | prob=0.9023
RENAME img0012.png -> Bird_of_Domination_0012.png | prob=0.9407
RENAME img0013.png -> Charred_Lizard_0004.png | prob=0.7376
RENAME img0014.png -> Lamp_Spirit_0006.png | prob=0.7844
RENAME img0015.png -> Necro_Princess_0011.png | prob=0.5497
RENAME img0016.png -> Rainbow_Pelican_EXP_0013.png | prob=0.9304
RENAME img0017.png -> Mysterious_Jar_0014.png | prob=0.8726